# ATLAS solar downscaling tutorial: reanalysis data

This notebook downscales monthly climatologies of **reanalysis solar radiation data** to the GLO-90 resolution (90m).

The workflow is designed for users with limited Python experience:

1. Edit only the **Input parameters** cell.
2. Run the **Functions** cells without changing them.
3. Run the final **Run downscaling** cell.

The notebook always works with **reanalysis data**.  
If `country == "argentina"`, the country is processed using the latitude-split method to reduce memory issues and avoid interpolation artefacts over a large domain.  
For all other countries, the full country is processed in one step.

## Methodology Description:

This notebook applies a Machine Learning based statistical downscaling approach to generate high-resolution climate fields from coarse-resolution climate datasets. The method relies on a Multi-Layer Perceptron (MLP) neural network trained to learn the relationship between large-scale climate variables and local terrain characteristics.

The downscaling procedure uses a set of predictors describing the influence of topography on local climate conditions. These predictors are derived from the Copernicus GLO90 Digital Elevation Model and typically include elevation, slope, aspect, and other terrain-related features. The target variable depends on the application and may include temperature, precipitation, solar radiation, or wind components.

During the training phase, the MLP is calibrated using the coarse-resolution climate variable together with the corresponding topographic predictors. The trained model is then applied to the high-resolution GLO90 grid, allowing the generation of climate information at a much finer spatial resolution.

The use of Earth Observation data is a key element of the methodology. High-resolution terrain information derived from satellite observations provides detailed spatial descriptors that are not represented in global climate datasets, enabling the neural network to reproduce local-scale spatial variability driven by topography.

This approach preserves the large-scale climate signal provided by reanalysis or climate model data while enhancing its spatial detail, producing high-resolution climate layers suitable for local impact assessments and climate adaptation studies.

In [1]:
# Required Python environment
# This notebook requires the same Python environment used by the ATLAS solar workflow.

from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


## Step 1. Input parameters

Edit this cell before running the notebook.

The input files are the outputs of the preprocessing notebooks.  
The DEM files (`orography` and `aspect`) are read from `../DEMdata/{country}/`.

The final output is saved in:

`../data/downscaling/{target}/{country}/`


In [2]:
# ---------------------------------------------------------------------
# USER INPUTS
# ---------------------------------------------------------------------

# Country name used in file and folder names.
# Use lower case to keep paths consistent, for example: "ecuador", "argentina".
country = "argentina"

# Month to downscale.
# Use an integer from 1 to 12.
month = 1

# Reanalysis target variable.
# For the solar reanalysis workflow this is usually "ssrd".
target = "ssrd"

# Period included in the preprocessed reanalysis files.
# Update these values only if your preprocessing files use a different date range in the filename.
start_date = "1991-01"
end_date = "2020-12"

# Input folders.
# These are the outputs from the preprocessing step.
processed_data_root = Path("../data/processed")

# DEM input folder.
# Orography and aspect are not read from the preprocessing folder.
dem_data_root = Path("../DEMdata")

# Final output folder.
output_path = Path(f"../data/downscaled_data/{target}/{country}")
output_path.mkdir(parents=True, exist_ok=True)

# Optional intermediate folder.
# This is only used internally if intermediate files need to be written.
intermediate_path = output_path / "intermediate_support_data"
intermediate_path.mkdir(parents=True, exist_ok=True)

# Automatic Argentina split settings.
# Keep these defaults unless you know that a different overlap is required.
split_argentina_by_latitude = True
argentina_split_overlap_deg = 1.0

# Input file paths from preprocessing.
# Change only the filenames if your preprocessing notebook produced different names.
input_paths = {
    "target": processed_data_root / "surface_solar_radiation_downwards" / country / f"{target}_{start_date}_{end_date}_processed.nc",
    "t2m": processed_data_root / "2m_temperature" / country / f"era5_t2m_{start_date}_{end_date}_processed.nc",
    "tcc": processed_data_root / "total_cloud_cover" / country / f"era5_tcc_{start_date}_{end_date}_processed.nc",
    "ghi": processed_data_root / "ghi" / country / f"ghi_2000-01_2026-03_processed.nc",
    "era5land_orography": dem_data_root / country / f"era5land_orography_{country}.nc",
    "era5land_aspect": dem_data_root / country / f"era5land_aspect_{country}.nc",
    "glo90_orography": dem_data_root / country / f"glo90_orography_{country}.nc",
    "glo90_aspect": dem_data_root / country / f"glo90_aspect_{country}.nc",
}

output_file = output_path / f"{target}_downscaled_{country}_m{month}.nc"

print("Country:", country)
print("Target:", target)
print("Month:", month)
print("Output file:", output_file)


Country: argentina
Target: ssrd
Month: 1
Output file: ../data/downscaled_data/ssrd/argentina/ssrd_downscaled_argentina_m1.nc


## Step 2. Utility functions

Run this section without editing it.


In [3]:
def check_input_files(paths):
    """Check that all required input files exist before starting the workflow."""
    missing = [name for name, path in paths.items() if not Path(path).exists()]
    if missing:
        missing_text = "\n".join(f"- {name}: {paths[name]}" for name in missing)
        raise FileNotFoundError(
            "Some required input files were not found. Please check the paths in the input cell:\n"
            f"{missing_text}"
        )


def drop_vars_if_present(ds, variables):
    """Drop variables only if they are present in the Dataset/DataArray."""
    existing = [var for var in variables if var in ds.variables]
    if existing:
        return ds.drop_vars(existing)
    return ds


def open_dataset_clean(path, reverse_latitude=True):
    """Open a NetCDF file and remove common technical variables if present."""
    ds = xr.open_mfdataset(str(path), chunks={})
    ds = drop_vars_if_present(ds, {"spatial_ref", "band"})
    if reverse_latitude and "latitude" in ds.coords:
        ds = ds.sel(latitude=slice(None, None, -1))
    return ds


def open_single_dataset_clean(path, reverse_latitude=False):
    """Open a single NetCDF file and remove common technical variables if present."""
    ds = xr.open_dataset(str(path), chunks={})
    ds = drop_vars_if_present(ds, {"spatial_ref", "band"})
    if reverse_latitude and "latitude" in ds.coords:
        ds = ds.sel(latitude=slice(None, None, -1))
    return ds


def save_xarray_netcdf_fast(ds, path):
    """Save an xarray Dataset as compressed NetCDF."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    encoding = {
        var: {
            "dtype": "float32",
            "zlib": True,
            "complevel": 1,
            "shuffle": True,
        }
        for var in ds.data_vars
    }

    temporary_path = path.with_name(path.stem + "_tmp.nc")
    if temporary_path.exists():
        temporary_path.unlink()

    ds.to_netcdf(
        temporary_path,
        engine="netcdf4",
        encoding=encoding,
        mode="w",
    )

    os.replace(temporary_path, path)


In [4]:
def era5_scaler(df, features, target_column):
    """Scale ERA5-Land features and the target variable."""
    feature_scaler = StandardScaler()
    X_scaled = feature_scaler.fit_transform(df[features])

    target_scaler = StandardScaler()
    y_scaled = target_scaler.fit_transform(df[[target_column]]).ravel()

    return target_scaler, X_scaled, y_scaled


def glo90_scaler(df, features):
    """Scale GLO-90 features before prediction."""
    scaler = StandardScaler()
    return scaler.fit_transform(df[features])


def merge_dataframe(left_df, right_df, variable):
    """Merge two dataframes using latitude and longitude as spatial keys."""
    return left_df.merge(
        right_df[["latitude", "longitude", variable]],
        on=["latitude", "longitude"],
        how="inner",
        validate="one_to_one",
    )


## Step 3. Functions for the reanalysis downscaling workflow

Run this section without editing it.


In [5]:
def make_training_dataset(target_ds, orography, aspect, t2m, tcc, ghi):
    """Create the low-resolution training dataframe from reanalysis and DEM inputs."""

    # Bring all predictors to the target grid.
    orography_interp = orography.interp(
        latitude=target_ds.latitude.values,
        longitude=target_ds.longitude.values,
        method="nearest",
    )
    aspect_interp = aspect.interp(
        latitude=target_ds.latitude.values,
        longitude=target_ds.longitude.values,
        method="nearest",
    )
    t2m_interp = t2m.interp(
        latitude=target_ds.latitude.values,
        longitude=target_ds.longitude.values,
        method="nearest",
    )
    tcc_interp = tcc.interp(
        latitude=target_ds.latitude.values,
        longitude=target_ds.longitude.values,
        method="nearest",
    )
    ghi_interp = ghi.interp(
        latitude=target_ds.latitude.values,
        longitude=target_ds.longitude.values,
        method="nearest",
    )

    # Monthly climatologies used for the month-specific model.
    target_monthly = target_ds.groupby(target_ds.time.dt.month).mean()
    t2m_monthly = t2m_interp.groupby(t2m_interp.time.dt.month).mean()
    tcc_monthly = tcc_interp.groupby(tcc_interp.time.dt.month).mean()
    ghi_monthly = ghi_interp.groupby(ghi_interp.time.dt.month).mean()

    merged = xr.merge([
        orography_interp,
        aspect_interp,
        t2m_monthly,
        tcc_monthly,
        ghi_monthly,
        target_monthly,
    ])

    return merged.to_dataframe().dropna().reset_index()


def train_downscaling_model(training_df, selected_month, target_column):
    """Train the month-specific neural-network downscaling model."""
    month_df = training_df.loc[training_df.month == selected_month, :].drop("month", axis=1)

    if "spatial_ref" in month_df.columns:
        month_df = month_df.drop("spatial_ref", axis=1)

    features = month_df.columns.drop([target_column])
    target_scaler, X_train, y_train = era5_scaler(month_df, features, target_column)

    model = MLPRegressor(random_state=1, max_iter=1000)
    model.fit(X_train, y_train)

    return features, target_scaler, model


def build_glo90_dataframe(orography_glo90, aspect_glo90, t2m_monthly, tcc_monthly, ghi_monthly):
    """Create the high-resolution feature dataframe used for prediction."""

    t2m_glo90 = t2m_monthly.interp(
        latitude=orography_glo90.latitude.values,
        longitude=orography_glo90.longitude.values,
        method="slinear",
    )
    tcc_glo90 = tcc_monthly.interp(
        latitude=orography_glo90.latitude.values,
        longitude=orography_glo90.longitude.values,
        method="slinear",
    )
    ghi_glo90 = ghi_monthly.interp(
        latitude=orography_glo90.latitude.values,
        longitude=orography_glo90.longitude.values,
        method="slinear",
    )
    aspect_glo90_interp = aspect_glo90.interp(
        latitude=orography_glo90.latitude.values,
        longitude=orography_glo90.longitude.values,
        method="nearest",
    )

    orography_df = drop_vars_if_present(orography_glo90, {"spatial_ref", "band"}).to_dataframe()
    aspect_df = drop_vars_if_present(aspect_glo90_interp, {"spatial_ref", "band"}).to_dataframe()

    orography_df = (
        orography_df.dropna().reset_index()
        .sort_values(["latitude", "longitude"], ascending=[False, True])
    )
    aspect_df = (
        aspect_df.dropna().reset_index()
        .sort_values(["latitude", "longitude"], ascending=[False, True])
    )

    df = merge_dataframe(orography_df, aspect_df, "aspect")

    t2m_df = (
        t2m_glo90.to_dataframe().dropna().reset_index()
        .sort_values(["latitude", "longitude"], ascending=[False, True])
    )
    df = merge_dataframe(df, t2m_df, "t2m")

    tcc_df = (
        tcc_glo90.to_dataframe().dropna().reset_index()
        .sort_values(["latitude", "longitude"], ascending=[False, True])
    )
    df = merge_dataframe(df, tcc_df, "tcc")

    ghi_df = (
        ghi_glo90.to_dataframe().dropna().reset_index()
        .sort_values(["latitude", "longitude"], ascending=[False, True])
    )
    df = merge_dataframe(df, ghi_df, "ghi")

    return df


def apply_downscaling(dataset_glo90, features, model, target_scaler, target_column):
    """Apply the trained model to the GLO-90 feature dataframe."""
    dataset_glo90 = dataset_glo90.copy()

    missing_features = [feature for feature in features if feature not in dataset_glo90.columns]
    if missing_features:
        raise ValueError(f"Missing GLO-90 features: {missing_features}")

    X_glo90 = glo90_scaler(dataset_glo90.loc[:, features], features)
    predicted_standardised = model.predict(X_glo90)
    predicted = target_scaler.inverse_transform(predicted_standardised.reshape(-1, 1))

    output_variable = f"{target_column}_downscaled"
    dataset_glo90[output_variable] = predicted

    output_ds = dataset_glo90.set_index(["latitude", "longitude"])[[output_variable]].to_xarray()

    # Convert ssrd from J m-2 to kWh m-2 day-1.
    # The factor follows the original reanalysis downscaling notebook.
    if target_column == "ssrd":
        output_ds[output_variable] = output_ds[output_variable] * 24 / 3_600_000

    return output_ds.sortby(["latitude", "longitude"])


## Step 4. Argentina split functions

Run this section without editing it.

These functions are used only when `country == "argentina"`.


In [6]:
def should_split_country(country_name):
    """Return True only for Argentina when the split option is enabled."""
    return split_argentina_by_latitude and country_name.lower() == "argentina"


def get_lat_split_from_coords(*datasets):
    """Compute the latitude split from coordinate extents only."""
    lat_min = min(float(ds["latitude"].min()) for ds in datasets if ds is not None)
    lat_max = max(float(ds["latitude"].max()) for ds in datasets if ds is not None)
    return (lat_min + lat_max) / 2


def _select_lat_range(ds, lat_min, lat_max):
    """Select a latitude range while preserving the native latitude order."""
    lat = ds["latitude"]
    if bool(lat[0] < lat[-1]):
        return ds.sel(latitude=slice(lat_min, lat_max))
    return ds.sel(latitude=slice(lat_max, lat_min))


def cut_by_lat(ds, lat_split, part, overlap_deg=0.0):
    """Cut a Dataset/DataArray into north or south latitude tiles with optional overlap."""
    full_lat_min = float(ds["latitude"].min())
    full_lat_max = float(ds["latitude"].max())

    if part == "north":
        return _select_lat_range(ds, max(full_lat_min, lat_split - overlap_deg), full_lat_max)
    if part == "south":
        return _select_lat_range(ds, full_lat_min, min(full_lat_max, lat_split + overlap_deg))

    raise ValueError("part must be 'north' or 'south'")


def cut_core_by_lat(ds, lat_split, part):
    """Remove the overlap before merging when blending is disabled."""
    full_lat_min = float(ds["latitude"].min())
    full_lat_max = float(ds["latitude"].max())

    if part == "north":
        return _select_lat_range(ds, lat_split, full_lat_max)
    if part == "south":
        return _select_lat_range(ds, full_lat_min, lat_split)

    raise ValueError("part must be 'north' or 'south'")


def merge_latitude_tiles(ds_south, ds_north, lat_split, overlap_deg=1.0, blend=True):
    """Merge south and north latitude tiles into one final country dataset."""
    ds_south = ds_south.sortby("latitude").sortby("longitude")
    ds_north = ds_north.sortby("latitude").sortby("longitude")

    if not blend:
        south_core = cut_core_by_lat(ds_south, lat_split, "south")
        north_core = cut_core_by_lat(ds_north, lat_split, "north")
        return xr.concat([south_core, north_core], dim="latitude", join="outer").sortby(["latitude", "longitude"])

    lat_union = np.union1d(ds_south.latitude.values, ds_north.latitude.values)
    lon_union = np.union1d(ds_south.longitude.values, ds_north.longitude.values)

    south_u = ds_south.reindex(latitude=lat_union, longitude=lon_union)
    north_u = ds_north.reindex(latitude=lat_union, longitude=lon_union)

    lat0 = lat_split - overlap_deg
    lat1 = lat_split + overlap_deg
    w_north = ((south_u.latitude - lat0) / (lat1 - lat0)).clip(0, 1)

    blended = south_u * (1 - w_north) + north_u * w_north
    full = blended.combine_first(south_u).combine_first(north_u)

    return full.dropna("latitude", how="all").dropna("longitude", how="all").sortby(["latitude", "longitude"])


## Step 5. Load data and run downscaling

Run this cell after checking the input parameters.


In [7]:
# Check that all required files are available.
check_input_files(input_paths)

# Load reanalysis target and feature datasets.
target_ds = open_dataset_clean(input_paths["target"])
t2m = open_dataset_clean(input_paths["t2m"])
tcc = open_dataset_clean(input_paths["tcc"])
ghi = open_dataset_clean(input_paths["ghi"])

# Load DEM and aspect datasets.
orography_era5land = open_single_dataset_clean(input_paths["era5land_orography"])
aspect_era5land = open_single_dataset_clean(input_paths["era5land_aspect"])
orography_glo90 = open_single_dataset_clean(input_paths["glo90_orography"])
aspect_glo90 = open_single_dataset_clean(input_paths["glo90_aspect"])

print("Input files loaded.")


Input files loaded.


ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


In [8]:
def process_downscaling_tile(tile_name=None, lat_split=None):
    """Train and apply the reanalysis downscaling workflow on one tile or on the full country."""

    if tile_name is None:
        target_tile = target_ds
        t2m_tile = t2m
        tcc_tile = tcc
        ghi_tile = ghi
        orography_era5land_tile = orography_era5land
        aspect_era5land_tile = aspect_era5land
        orography_glo90_tile = orography_glo90
        aspect_glo90_tile = aspect_glo90
    else:
        target_tile = cut_by_lat(target_ds, lat_split, tile_name, argentina_split_overlap_deg)
        t2m_tile = cut_by_lat(t2m, lat_split, tile_name, argentina_split_overlap_deg)
        tcc_tile = cut_by_lat(tcc, lat_split, tile_name, argentina_split_overlap_deg)
        ghi_tile = cut_by_lat(ghi, lat_split, tile_name, argentina_split_overlap_deg)
        orography_era5land_tile = cut_by_lat(orography_era5land, lat_split, tile_name, argentina_split_overlap_deg)
        aspect_era5land_tile = cut_by_lat(aspect_era5land, lat_split, tile_name, argentina_split_overlap_deg)
        orography_glo90_tile = cut_by_lat(orography_glo90, lat_split, tile_name, argentina_split_overlap_deg)
        aspect_glo90_tile = cut_by_lat(aspect_glo90, lat_split, tile_name, argentina_split_overlap_deg)

    training_df = make_training_dataset(
        target_tile,
        orography_era5land_tile,
        aspect_era5land_tile,
        t2m_tile,
        tcc_tile,
        ghi_tile,
    )

    features, target_scaler, model = train_downscaling_model(training_df, month, target)

    t2m_monthly = t2m_tile.groupby(t2m_tile.time.dt.month).mean().sel(month=month)
    tcc_monthly = tcc_tile.groupby(tcc_tile.time.dt.month).mean().sel(month=month)
    ghi_monthly = ghi_tile.groupby(ghi_tile.time.dt.month).mean().sel(month=month)

    t2m_monthly = drop_vars_if_present(t2m_monthly, {"month"})
    tcc_monthly = drop_vars_if_present(tcc_monthly, {"month"})
    ghi_monthly = drop_vars_if_present(ghi_monthly, {"month"})

    glo90_df = build_glo90_dataframe(
        orography_glo90_tile,
        aspect_glo90_tile,
        t2m_monthly,
        tcc_monthly,
        ghi_monthly,
    )

    return apply_downscaling(glo90_df, features, model, target_scaler, target)


if should_split_country(country):
    lat_split = get_lat_split_from_coords(target_ds, t2m, tcc, ghi)

    north_ds = process_downscaling_tile("north", lat_split=lat_split)
    south_ds = process_downscaling_tile("south", lat_split=lat_split)

    output_ds = merge_latitude_tiles(
        south_ds,
        north_ds,
        lat_split=lat_split,
        overlap_deg=argentina_split_overlap_deg,
        blend=True,
    )
else:
    output_ds = process_downscaling_tile()

save_xarray_netcdf_fast(output_ds, output_file)

print("Downscaling completed.")
print("Saved output:", output_file)


Downscaling completed.
Saved output: ../data/downscaled_data/ssrd/argentina/ssrd_downscaled_argentina_m1.nc
